# Docent DQL Notebook
Fetches agent runs via DQL and loads them into pandas.

### README
- Requires `pip install --upgrade docent-python pandas`
- API key is prefilled; rotate in Docent if you regenerate keys.
- Run the cells as-is to fetch data into pandas.


In [ ]:
%pip install -q --upgrade docent-python pandas


In [ ]:
from docent import Docent
import pandas as pd

API_KEY = "dk_Dufq4mImYrRtFNcv_ecle7OfxInL72bgjpHFe9tph55DMtclTsMkAPeGVAQk0XG"
SERVER_URL = "https://api.docent-bridgewater.transluce.org"
COLLECTION_ID = "07287acb-8908-4f4c-95ec-8cdeadcb4423"

DQL_QUERY = """
SELECT
  ar.id AS agent_run_id,
  ar.metadata_json->>'as_of_date' AS metadata_as_of_date,
  ar.metadata_json->>'base_llm_name' AS metadata_base_llm_name,
  ar.metadata_json->>'brier_score' AS metadata_brier_score,
  ar.metadata_json->>'calibration_llm_name' AS metadata_calibration_llm_name,
  ar.metadata_json->>'consensus_llm_name' AS metadata_consensus_llm_name,
  ar.metadata_json->>'experiment_id' AS metadata_experiment_id,
  ar.metadata_json->'experimental_config'->>'baseline_forecast' AS metadata_experimental_config_baseline_forecast,
  ar.metadata_json->'experimental_config'->>'baseline_forecast_explanation' AS metadata_experimental_config_baseline_forecast_explanation,
  ar.metadata_json->'experimental_config'->>'check_vibe_hedging' AS metadata_experimental_config_check_vibe_hedging,
  ar.metadata_json->'experimental_config'->>'llm_reasoning_effort' AS metadata_experimental_config_llm_reasoning_effort,
  ar.metadata_json->'experimental_config'->>'llm_verbosity' AS metadata_experimental_config_llm_verbosity,
  ar.metadata_json->'experimental_config'->>'resolution_criteria' AS metadata_experimental_config_resolution_criteria,
  ar.metadata_json->'experimental_config'->>'resolution_date' AS metadata_experimental_config_resolution_date,
  ar.metadata_json->>'final_probability' AS metadata_final_probability,
  ar.metadata_json->>'finished' AS metadata_finished,
  ar.metadata_json->>'forecast_id' AS metadata_forecast_id,
  ar.metadata_json->>'llm_reasoning_effort' AS metadata_llm_reasoning_effort,
  ar.metadata_json->>'llm_verbosity' AS metadata_llm_verbosity,
  ar.metadata_json->>'meta_llm_name' AS metadata_meta_llm_name,
  ar.metadata_json->>'model' AS metadata_model,
  ar.metadata_json->>'post_consistency_probability' AS metadata_post_consistency_probability, // consistency
  ar.metadata_json->>'post_platt_scaling_probability' AS metadata_post_platt_scaling_probability, //scaled
  ar.metadata_json->>'pre_consistency_confidence_interval_high' AS metadata_pre_consistency_confidence_interval_high,
  ar.metadata_json->>'pre_consistency_confidence_interval_low' AS metadata_pre_consistency_confidence_interval_low,
  ar.metadata_json->>'pre_consistency_mean_probability' AS metadata_pre_consistency_mean_probability,  // mean
  ar.metadata_json->>'pre_consistency_variance' AS metadata_pre_consistency_variance,
  ar.metadata_json->>'pre_platt_scaling_probability' AS metadata_pre_platt_scaling_probability,
  ar.metadata_json->>'question' AS metadata_question,
  ar.metadata_json->>'resolved_to' AS metadata_resolved_to,  // resolved_to
  ar.created_at AS created_at
FROM agent_runs ar
ORDER BY ar.created_at ASC
"""

client = Docent(api_key=API_KEY, server_url=SERVER_URL)
result = client.execute_dql(COLLECTION_ID, DQL_QUERY)
df = client.dql_result_to_df_experimental(result)

df.head()


In [ ]:
# Convert columns to numeric
df['metadata_resolved_to'] = pd.to_numeric(df['metadata_resolved_to'], errors='coerce')
df['metadata_pre_consistency_mean_probability'] = pd.to_numeric(df['metadata_pre_consistency_mean_probability'], errors='coerce')
df['metadata_post_consistency_probability'] = pd.to_numeric(df['metadata_post_consistency_probability'], errors='coerce')
df['metadata_post_platt_scaling_probability'] = pd.to_numeric(df['metadata_post_platt_scaling_probability'], errors='coerce')

# Compute Brier scores as columns: (forecast - outcome)^2
df['brier_score_mean'] = (df['metadata_pre_consistency_mean_probability'] - df['metadata_resolved_to']) ** 2
df['brier_score_consistency'] = (df['metadata_post_consistency_probability'] - df['metadata_resolved_to']) ** 2
df['brier_score_scaled'] = (df['metadata_post_platt_scaling_probability'] - df['metadata_resolved_to']) ** 2

# Display summary statistics
print("Brier Score Summary:")
print(f"  Mean probability:        {df['brier_score_mean'].mean():.6f}")
print(f"  Consistency probability: {df['brier_score_consistency'].mean():.6f}")
print(f"  Scaled probability:      {df['brier_score_scaled'].mean():.6f}")

df[['agent_run_id', 'brier_score_mean', 'brier_score_consistency', 'brier_score_scaled']].head()



In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt

# Group by metadata_base_llm_name and metadata_resolved_to, compute mean Brier scores
scores_by_llm_outcome = df.groupby(['metadata_base_llm_name', 'metadata_resolved_to']).agg({
    'brier_score_mean': 'mean',
    'brier_score_consistency': 'mean',
    'brier_score_scaled': 'mean'
}).reset_index()

# Create separate dataframes for resolved_to = 0 and resolved_to = 1
scores_resolved_0 = scores_by_llm_outcome[scores_by_llm_outcome['metadata_resolved_to'] == 0].copy()
scores_resolved_1 = scores_by_llm_outcome[scores_by_llm_outcome['metadata_resolved_to'] == 1].copy()

# Sort by mean score for better visualization
scores_resolved_0 = scores_resolved_0.sort_values('brier_score_mean')
scores_resolved_1 = scores_resolved_1.sort_values('brier_score_mean')

# Create subplots - one for each outcome
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot for resolved_to = 0
ax0 = axes[0]
x0 = range(len(scores_resolved_0))
width = 0.25

bars1_0 = ax0.bar([i - width for i in x0], scores_resolved_0['brier_score_mean'], width, label='Mean Probability', color='steelblue')
bars2_0 = ax0.bar(x0, scores_resolved_0['brier_score_consistency'], width, label='Consistency Probability', color='darkorange')
bars3_0 = ax0.bar([i + width for i in x0], scores_resolved_0['brier_score_scaled'], width, label='Scaled Probability', color='forestgreen')

for bar in bars1_0:
    height = bar.get_height()
    ax0.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)

for bar in bars2_0:
    height = bar.get_height()
    ax0.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)

for bar in bars3_0:
    height = bar.get_height()
    ax0.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)

ax0.set_xlabel('Base LLM Name')
ax0.set_ylabel('Brier Score (lower is better)')
ax0.set_title('Brier Scores by Base LLM (Resolved to 0 / No)')
ax0.set_xticks(x0)
ax0.set_xticklabels(scores_resolved_0['metadata_base_llm_name'], rotation=45, ha='right')
ax0.legend()

# Plot for resolved_to = 1
ax1 = axes[1]
x1 = range(len(scores_resolved_1))

bars1_1 = ax1.bar([i - width for i in x1], scores_resolved_1['brier_score_mean'], width, label='Mean Probability', color='steelblue')
bars2_1 = ax1.bar(x1, scores_resolved_1['brier_score_consistency'], width, label='Consistency Probability', color='darkorange')
bars3_1 = ax1.bar([i + width for i in x1], scores_resolved_1['brier_score_scaled'], width, label='Scaled Probability', color='forestgreen')

for bar in bars1_1:
    height = bar.get_height()
    ax1.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)

for bar in bars2_1:
    height = bar.get_height()
    ax1.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)

for bar in bars3_1:
    height = bar.get_height()
    ax1.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)

ax1.set_xlabel('Base LLM Name')
ax1.set_ylabel('Brier Score (lower is better)')
ax1.set_title('Brier Scores by Base LLM (Resolved to 1 / Yes)')
ax1.set_xticks(x1)
ax1.set_xticklabels(scores_resolved_1['metadata_base_llm_name'], rotation=45, ha='right')
ax1.legend()

plt.tight_layout()
plt.show()

# Print summary comparison
print("\nBrier Score Comparison by Outcome:")
print("\nResolved to 0 (No):")
print(f"  Mean probability:        {scores_resolved_0['brier_score_mean'].mean():.6f}")
print(f"  Consistency probability: {scores_resolved_0['brier_score_consistency'].mean():.6f}")
print(f"  Scaled probability:      {scores_resolved_0['brier_score_scaled'].mean():.6f}")

print("\nResolved to 1 (Yes):")
print(f"  Mean probability:        {scores_resolved_1['brier_score_mean'].mean():.6f}")
print(f"  Consistency probability: {scores_resolved_1['brier_score_consistency'].mean():.6f}")
print(f"  Scaled probability:      {scores_resolved_1['brier_score_scaled'].mean():.6f}")


In [ ]:
# Create pivot table comparing o3 and gpt-5 scaled Brier scores by question
filtered_df = df[df['metadata_base_llm_name'].isin(['o3', 'gpt-5'])]

# Pivot table for mean scaled Brier scores
pivot_df = filtered_df.pivot_table(
    index='metadata_question',
    columns='metadata_base_llm_name',
    values='brier_score_scaled',
    aggfunc='mean'
)[['o3', 'gpt-5']]

pivot_df.columns = ['o3_mean_scaled_brier', 'gpt-5_mean_scaled_brier']

# Pivot table for agent_run_ids (as lists)
agent_run_ids_pivot = filtered_df.pivot_table(
    index='metadata_question',
    columns='metadata_base_llm_name',
    values='agent_run_id',
    aggfunc=list
)[['o3', 'gpt-5']]

agent_run_ids_pivot.columns = ['o3_agent_run_ids', 'gpt-5_agent_run_ids']

# Merge the two pivot tables
pivot_df = pivot_df.join(agent_run_ids_pivot)

pivot_df['gpt5_minus_o3'] = pivot_df['gpt-5_mean_scaled_brier'] - pivot_df['o3_mean_scaled_brier']
pivot_df = pivot_df.sort_values('gpt5_minus_o3', ascending=False)
pivot_df


In [ ]:
pivot_df.iloc[:3].to_dict()